# Fig. S8: calibration of the class thresholds

a: GOE revival null (median, 99th percentile) against subspace dimension D,
with the recurrence boundary D*. b: 99th percentile / median of the null
against the anomaly cut. c: median F_rev of real subspaces (N = 12) against
the null. d: anomaly ratio of real states above D*. e: floor ratio of the
scar population.

Data: `data/revival_null.npz`, `data/revival_null_early_dip.npz`,
`data/classes_N12_*.h5`, `data/relax_diag_N12_*.npz`.

In [ ]:
%matplotlib inline

import math

import h5py

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qising_networks import DATA_DIR as RESULTS, FIGURES_DIR as FIGURES
from qising_networks import PROJECT_ROOT
from qising_networks import fig_maker as fm
from qising_networks import scar_metrics as sm
from qising_networks import spectra

if (PROJECT_ROOT / "matplotlibrc").exists():
    mpl.rc_file(PROJECT_ROOT / "matplotlibrc")

FIGURES.mkdir(exist_ok=True)
NULL_PATH = RESULTS / "revival_null.npz"
N_SPINS = 12
MODELS = (("ferro", "Ferromagnetic"), ("pmJ", r"$\pm J$"))
REV_CUT = 0.1
ANOMALY_CUT = 3.0
FLOOR_CUT = 0.1


In [ ]:
def d_star_crossing(D: np.ndarray, p50: np.ndarray, cut: float = REV_CUT) -> float:
    """Log-log interpolate the D where the GOE median crosses `cut`."""
    for i in range(len(D) - 1):
        if (p50[i] - cut) * (p50[i + 1] - cut) <= 0:
            f = (math.log(cut) - math.log(p50[i])) / (
                math.log(p50[i + 1]) - math.log(p50[i])
            )
            return math.exp(
                math.log(D[i]) + f * (math.log(D[i + 1]) - math.log(D[i]))
            )
    raise RuntimeError(f"median never crosses {cut}")


z = np.load(NULL_PATH)
t = z["table"]
D, p50, p99 = t[:, 0], t[:, 3], t[:, 5]
ratio = p99 / p50
D_star = d_star_crossing(D, p50)
print(f"D_* = {D_star:.2f}")

ratio_above = ratio[D >= D_star]
print(f"99th/median: all D {ratio.min():.2f}-{ratio.max():.2f}; "
      f"D >= D_* {ratio_above.min():.2f}-{ratio_above.max():.2f}")
for d in (4, 68, 392):
    i = int(np.argmin(abs(D - d)))
    print(f"null median at D={D[i]:.0f}: {p50[i]:.3f}")
print(f"samples per D: {t[:, 2].max():.0f} (D <= 64) down to "
      f"{t[:, 2].min():.0f} (D >= {D[t[:, 2] == t[:, 2].min()].min():.0f})")


In [ ]:
# Time-grid convergence of the null at D = 32 (quoted in the supplement).
GRID_CHECK_D = 32
GRID_CHECK_POINTS = (400, 800, 1600, 3200)
GRID_CHECK_REPS = 200
GRID_CHECK_SEED = 0


def null_median_on_grid(dim, n_points, reps, seed):
    extent = float(np.interp(np.log(dim), np.log(t[:, 0]), t[:, 1]))
    time_max = float(z["time_max"])
    rng = np.random.default_rng(seed)
    tt = np.linspace(0.0, time_max, n_points)
    vals = []
    for _ in range(reps):
        g = rng.normal(size=(dim, dim))
        ev, U = np.linalg.eigh((g + g.T) / 2.0)
        ev = ev * extent / (ev[-1] - ev[0])
        a = int(rng.integers(dim))
        F = np.abs((U[a] ** 2) @ np.exp(-1j * np.outer(ev, tt))) ** 2
        vals.append(float(sm.revival(F, tt)))
    return float(np.median(vals))


grid_check = {n: null_median_on_grid(GRID_CHECK_D, n, GRID_CHECK_REPS,
                                     GRID_CHECK_SEED)
              for n in GRID_CHECK_POINTS}
print(f"D = {GRID_CHECK_D}, null median F_rev vs grid points: "
      + ", ".join(f"{n}: {v:.3f}" for n, v in grid_check.items()))


In [ ]:
# Per-state revivals of the N = 12 class-fraction data; anomaly ratio per
# realisation, as in the classifier.
def load_states(model):
    path = RESULTS / f"classes_N{N_SPINS}_{model}.h5"
    fmin, frev, size, cls, anom = [], [], [], [], []
    with h5py.File(path) as f:
        for g in f["realisations"].values():
            s = g["states"][:]
            fm, fr, ne, cs = s[:, 0], s[:, 1], s[:, 2], s[:, 4]
            fmin.append(fm)
            frev.append(fr)
            size.append(cs)
            cls.append(sm.classify(fm, fr, ne, N_SPINS, comp_size=cs))
            anom.append(sm.anomaly_ratio(fr, cs))
    return {k: np.concatenate(v) for k, v in zip(
        ("fmin", "frev", "size", "cls", "anom"),
        (fmin, frev, size, cls, anom))}


real = {m: load_states(m) for m, _ in MODELS}
D_REC = 180  # reported recurrence boundary (D_* rounded)
BINS = np.unique(np.round(np.geomspace(2, 2100, 25)).astype(int))
for m, label in MODELS:
    r = real[m]
    idx = np.digitize(r["size"], BINS)
    Dc, med = [], []
    for b in range(1, len(BINS)):
        sel = idx == b
        if sel.sum() >= 200:
            Dc.append(np.median(r["size"][sel]))
            med.append(np.median(r["frev"][sel]))
    r["Dc"], r["med"] = np.array(Dc), np.array(med)
    r["goe"] = np.exp(np.interp(np.log(r["Dc"]), np.log(D), np.log(p50)))
    q = r["med"] / r["goe"]
    above = r["size"] >= D_REC
    sc = (r["cls"] == sm.SCAR) | (r["cls"] == sm.PLATEAU_SCAR)
    r["floor"] = r["fmin"][sc] / r["frev"][sc]
    print(f"{label}: {len(r['size'])} states; real/GOE median "
          f"{q.min():.2f}-{q.max():.2f} "
          f"(D~{r['Dc'][np.argmin(q)]:.0f} to D~{r['Dc'][np.argmax(q)]:.0f}); "
          f"at D~16 {q[np.argmin(abs(r['Dc'] - 16))]:.2f}, "
          f"D~1000 {q[np.argmin(abs(r['Dc'] - 1000))]:.2f}")
    print(f"    D >= {D_REC}: {above.sum()} states, anomaly >= 3: "
          f"{100 * (r['anom'][above] >= ANOMALY_CUT).mean():.1f}%")
    print(f"    scar + plateau scar: {sc.sum()}; floor ratio < 0.1: "
          f"{100 * (r['floor'] < 0.1).mean():.0f}%, max {r['floor'].max():.2f}")


In [ ]:
with plt.rc_context({'font.size': 7, 'legend.fontsize': 5}):
    fig = plt.figure(figsize=(fm.MAX_FIG_WIDTH, 4.6), constrained_layout=True)
    gs = fig.add_gridspec(2, 6)
    axes = [fig.add_subplot(gs[0, 0:2]), fig.add_subplot(gs[0, 2:4]),
            fig.add_subplot(gs[0, 4:6]), fig.add_subplot(gs[1, 0:3]),
            fig.add_subplot(gs[1, 3:6])]
    MCOL = {"ferro": "C0", "pmJ": "C1"}

    # a: GOE null and the recurrence boundary.
    ax = axes[0]
    ax.loglog(D, p50, "o-", ms=2.5, lw=0.8, color="k", label="Median")
    ax.loglog(D, p99, "s--", ms=2.0, lw=0.8, color="0.5",
              label=r"$99\mathrm{th}$ percentile")
    ax.axhline(REV_CUT, color="black", ls=":", lw=0.8)
    ax.axvline(D_star, color="black", ls=":", lw=0.8)
    ax.text(D_star * 1.15, 0.5, rf"$D^\ast \approx {D_star:.0f}$")
    ax.set_xlabel(r"Subspace dimension $D$")
    ax.set_ylabel(r"GOE null $F_{\mathrm{rev}}(\boldsymbol{s}_0)$")
    ax.set_xlim(2, 5000)
    ax.set_ylim(2e-3, 1.5)
    ax.legend(frameon=False, loc="lower left")

    # b: chance spread of the null against the anomaly cut.
    ax = axes[1]
    ax.axvspan(2, D_star, color="0.9", lw=0)
    ax.text(3, 2.6, "Recurrence\nregime", va="top")
    ax.semilogx(D, ratio, "o-", ms=2.5, lw=0.8, color="k")
    ax.axhline(ANOMALY_CUT, color="C3", lw=0.9, label="Threshold $3$")
    ax.axhline(ratio_above.max(), color="black", ls=":", lw=0.8,
               label=rf"Max for $D \geq D^\ast$ (${ratio_above.max():.2f}$)")
    ax.set_xlabel(r"Subspace dimension $D$")
    ax.set_ylabel(r"GOE null $99\mathrm{th}$/median")
    ax.set_xlim(2, 5000)
    ax.set_ylim(0.9, 3.4)
    ax.legend(frameon=False, loc="center right")

    # c: real subspaces recur more than the GOE null.
    ax = axes[2]
    ax.loglog(D, p50, "-", lw=0.8, color="k", label="GOE null median")
    for m, label in MODELS:
        r = real[m]
        ax.loglog(r["Dc"], r["med"], "o", ms=2.5, color=MCOL[m],
                  label=f"{label}, $N = {N_SPINS}$")
    ax.axvline(D_REC, color="black", ls=":", lw=0.8)
    ax.set_xlabel(r"Subspace dimension $D$")
    ax.set_ylabel(r"Median $F_{\mathrm{rev}}(\boldsymbol{s}_0)$")
    ax.set_xlim(2, 5000)
    ax.set_ylim(2e-3, 1.5)
    ax.legend(frameon=False, loc="lower left")

    # d: anomaly ratio of real states above the recurrence boundary.
    ax = axes[3]
    bins = np.linspace(0, 12, 61)
    for m, label in MODELS:
        r = real[m]
        a = r["anom"][r["size"] >= D_REC]
        frac = 100 * (a >= ANOMALY_CUT).mean()
        ax.hist(a, bins=bins, histtype="step",
                color=MCOL[m], lw=0.8, density=True,
                label=f"{label} (${frac:.1f}\\%$ at $\\geq 3$)")
    ax.axvline(ANOMALY_CUT, color="C3", lw=0.9)
    ax.set_yscale("log")
    ax.set_xlabel(r"Anomaly $F_{\mathrm{rev}}(\boldsymbol{s}_0)"
                  r"/\mathrm{med}\,F_{\mathrm{rev}}$"
                  rf" ($D \geq {D_REC}$)")
    ax.set_ylabel("Probability density")
    ax.set_xlim(0, 12)
    ax.legend(frameon=False, loc="upper right")

    # e: floor ratio of the scar population.
    ax = axes[4]
    bins = np.linspace(0, 1, 41)
    for m, label in MODELS:
        ax.hist(real[m]["floor"], bins=bins, histtype="step",
                color=MCOL[m], lw=0.8, density=True, label=label)
    ax.axvline(FLOOR_CUT, color="C3", lw=0.9, label="Threshold $0.1$")
    ax.set_yscale("log")
    ax.set_xlabel(r"Floor ratio $\min_t F(t; \boldsymbol{s}_0)"
                  r"/F_{\mathrm{rev}}(\boldsymbol{s}_0)$ (scar states)")
    ax.set_ylabel("Probability density")
    ax.set_xlim(0, 1)
    ax.legend(frameon=False, loc="upper right")

    fm.align_panel_labels_to_yaxis(
        fig, axes, fm.journal_labels("abcde"), fontsize=8, bold=True)
    first = [axes[0], axes[3]]
    x_left = min(ax.transAxes.transform(ax.texts[-1].get_position())[0]
                 for ax in first)
    for ax in first:
        ax.texts[-1].set_x(ax.transAxes.inverted().transform((x_left, 0))[0])


In [ ]:
paths = fm.save_unlabeled(fig, FIGURES / 'fig-s08', 'fig-s08')
print(paths)


## Numbers quoted in the supplement: revival after the relaxation time

The side lobe of the early-dip null (`revival_null_early_dip.npz`, the
first collapse searched for in Jt <= 50) and the N = 12
comparison with the first-quarter rule (`scripts/diagnose_relax_time.py`).

In [ ]:
from scipy.special import j1

# first side lobe of the semicircle decay [2 J1(x) / x]^2
x = np.linspace(1e-3, 12.0, 200001)
lobe = (2 * j1(x) / x) ** 2
zeros = x[1:-1][(lobe[1:-1] < lobe[:-2]) & (lobe[1:-1] < lobe[2:])]
SIDE_LOBE = lobe[(x > zeros[0]) & (x < zeros[1])].max()
old = np.load(RESULTS / "revival_null_early_dip.npz")["table"]
big = old[:, 0] >= 1000
print(f"first side lobe {SIDE_LOBE:.4f}; early-dip null (t_min in Jt <= 50), D >= 1000: "
      f"p99 {old[big, 5].min():.4f}-{old[big, 5].max():.4f}, median {old[big, 3].min():.4f}-{old[big, 3].max():.4f}")


In [ ]:
for model in ("ferro", "pmJ"):
    d = np.load(RESULTS / f"relax_diag_N12_{model}.npz", allow_pickle=True)
    cq, cr = d["cls_quarter"], d["cls_relax"]
    nq, nr = d["counts_quarter"].sum(axis=0), d["counts_relax"].sum(axis=0)
    gain = np.flatnonzero((cq == sm.THERMALIZED) & (cr == sm.SCAR))
    loss = np.flatnonzero((cq == sm.SCAR) & (cr == sm.THERMALIZED))
    pre, t_peak, t_dip = [], [], []
    for i in gain:
        F, t = d["traces"][i], d["times"][i]
        j = np.searchsorted(t, sm.RELAX_TIME)
        pre.append(F[:j].min())
        t_peak.append(t[j + np.argmax(F[j:])])
        t_dip.append(t[np.argmin(F[: sm._collapse_window(t, None)])])
    lower = d["frev_relax"][loss] < d["frev_quarter"][loss] * (1 - 1e-9)
    same = (nq[[sm.LOCALIZED, sm.POINCARE]] == nr[[sm.LOCALIZED, sm.POINCARE]]).all()
    print(f"{model}: {nq.sum()} states; localized/recurrence identical: {same}; "
          f"scars {nq[sm.SCAR]} -> {nr[sm.SCAR]}")
    print(f"  thermalized -> scar {len(gain)}: median min F before t_relax {np.median(pre):.3f}, "
          f"median peak Jt {np.median(t_peak):.0f}, median first-quarter dip Jt {np.median(t_dip):.0f}")
    print(f"  scar -> thermalized {len(loss)}: anomaly median {np.median(d['anom_quarter'][loss]):.2f} -> "
          f"{np.median(d['anom_relax'][loss]):.2f}; largest fidelity before t_relax: {lower.sum()}")


In [ ]:
# Time-grid sizes of the null at D = 32 and 180.
from qising_networks.scar_metrics import make_time_grid
null_npz = np.load(NULL_PATH)
null_tab = null_npz["table"]
for D in (32, 180):
    ext = float(np.interp(np.log(D), np.log(null_tab[:, 0]), null_tab[:, 1]))
    grid, _w, _o = make_time_grid(np.array([0.0, ext]),
                                  float(null_npz["time_max"]))
    print(f"D = {D}: bandwidth {ext:.3f}, {len(grid)} grid points")
real_dims = null_npz["real_dims"]
print(f"bandwidth source: {len(np.unique(real_dims))} distinct dimensions, "
      f"largest {int(real_dims.max())}")


In [ ]:
# <r> of small connected subspaces (the networks of the null, degeneracies
# lifted by sector_diagonal_noise) and of GOE matrices of the same sizes.

from qising_networks.model import build_network, degree_to_h_on_off, h_on_off_to_h, sector_diagonal_noise
from qising_networks.model import GAMMA, largest_sector


def mean_r(ev):
    return float(spectra.level_spacing_ratios(ev).mean())


small_r = {"8-12": [], "32-64": []}
n_lv = {"8-12": [], "32-64": []}
for net in range(12):
    rng = np.random.default_rng(net)
    im, degree = build_network(13, "ferro", rng)
    h_on, h_off = degree_to_h_on_off(degree)
    for k in range(14):
        h = h_on_off_to_h(h_on, h_off, k, rng)
        sec, H = largest_sector(im, h, GAMMA)
        if sec is None or len(sec) < 2:
            continue
        ss = sm.states_from_indices(sec, 13)
        lab = sm.split_components(sm.adjacency_from_H(H, GAMMA), ss).labels
        H = H + np.diag(sector_diagonal_noise(
            sec, 13, GAMMA, np.random.default_rng([net, k, 7])))
        for c in np.unique(lab):
            mem = np.flatnonzero(lab == c)
            key = ("8-12" if 8 <= len(mem) <= 12 else
                   "32-64" if 32 <= len(mem) <= 64 else None)
            if key:
                lv = np.linalg.eigvalsh(H[np.ix_(mem, mem)])
                if len(lv) >= 3:
                    small_r[key].append(mean_r(lv))
                    n_lv[key].append(len(lv))
goe_rng = np.random.default_rng(0)
for key in small_r:
    goe = [mean_r(np.linalg.eigvalsh((g + g.T) / 2))
           for D in np.unique(n_lv[key])
           for g in [goe_rng.normal(size=(D, D)) for _ in range(200)]]
    print(f"D = {key}: {len(small_r[key])} subspaces, mean <r> = "
          f"{np.mean(small_r[key]):.3f} (GOE of the same sizes {np.mean(goe):.3f}), "
          f"levels {min(n_lv[key])}-{max(n_lv[key])}")
